## 실습
1. 데이터 구축
2. 컬렉션 만들고 데이터 적재
3. 검색 테스트
4. ChromaDB 저장하기
5. 불러와서 테스트
6. context에 포함하여 최종 요청

### 1. 데이터 구축

In [42]:
import pandas as pd

df = pd.read_csv("../data/한국언론진흥재단_뉴스빅데이터_메타데이터_범죄보도_20231231.csv")

df['일자']

0        2016-01-03
1        2016-01-03
2        2016-01-03
3        2016-01-03
4        2016-01-03
            ...    
81308    2023-10-14
81309    2023-10-14
81310    2023-10-14
81311    2023-10-14
81312    2023-10-14
Name: 일자, Length: 81313, dtype: str

In [43]:
df.isna().sum()

제목                 0
본문               230
언론사                0
일자                 0
사건_사고 분류           0
분류 키워드             0
폭력_강도_살인 범죄건수      0
성 범죄건수             0
이상동기 범죄건수          0
dtype: int64

In [47]:
df = df.dropna(subset="본문").reset_index()
df.isna().sum()

index            0
제목               0
본문               0
언론사              0
일자               0
사건_사고 분류         0
분류 키워드           0
폭력_강도_살인 범죄건수    0
성 범죄건수           0
이상동기 범죄건수        0
dtype: int64

In [48]:
mask = ['제목', '본문', '일자']

df = df[mask]
df.head()

,제목,본문,일자
0,"해병대 ""폭행·가혹행위, 적법하게 처리"" 은폐 의혹 부인",해병대사령부는 최전방 해병대 간부들이 부대 밖에서 술을 마시다가 후임을 집단으로 폭...,2016-01-03
1,"법원 ""'성추행·보험 사기' 경찰 해임은 적법""","간호사를 성추행하고, 보험 사기를 친 경찰을 해임한 처분은 적법하다는 법원 판결이 ...",2016-01-03
2,"이천 '빗자루 교사 폭행' 가해학생, SNS에 ""교사가 맞을 짓 했다""",경기 이천의 고교생들이 수업시간에 30대 교사를 빗자루로 폭행해 형사 입건된 가운데...,2016-01-03
3,"엄마에 치근덕대다, 딸 강간한 미친 배달원",한 음식점 배달원이 길에서 만난 한 여성에게 “한잔 하자”며 치근덕대다 거절당한 뒤...,2016-01-03
4,백령도 해병대 집단 폭행·가혹 행위 물의,"백령도 해병대 간부들이 부대 밖에서 술을 마시다가 후임을 집단 폭행하고, 부대 안에...",2016-01-03


2016-01-03부터 2023-10-14까지의 범죄 기사가 총 81313개 있으니, 이 중 800개만 균일하게 뽑자 -> 년수 8년에 대량 해 당 10000개 -> 해 당 100개씩만 뽑기!

In [ ]:
# 일자에서 연도를 만든 뒤, 연도별로 100 -> 50 -> 30개씩 무작위 추출
df_sampled = (
    df.assign(연도=pd.to_datetime(df['일자']).dt.year)
      .groupby('연도', group_keys=False)
      .sample(n=30, random_state=42)
      .sort_values('일자')
      .reset_index(drop=True)
)

# 연도마다 정확히 100개씩, 총 800개인지 확인
print(df_sampled['연도'].value_counts().sort_index())
print(f'전체 개수: {len(df_sampled)}')

df_sampled.head()

연도
2016    30
2017    30
2018    30
2019    30
2020    30
2021    30
2022    30
2023    30
Name: count, dtype: int64
전체 개수: 240


,제목,본문,일자,연도
0,술 잘 마시던 형제가 왜 서로 흉기를 들이댔을까?,7일 오후 9시20분쯤 전북 전주시 우아동의 한 자택에서 40대 남성 2명이 흉기에...,2016-01-08,2016
1,"40대 가장, 부인·자녀 등 3명 살해한 뒤 투신",경기 광주시에 있는 아파트에서 40대 가장이 부인과 자녀 2명을 살해한 뒤 스스로 ...,2016-01-21,2016
2,"'횡령 의혹' 장애인시설 원장과 가족, 무혐의 처분",횡령과 폭행 의혹을 받았던 경기도 남양주시의 한 장애인 시설 원장과 가족이 수사를 ...,2016-02-17,2016
3,건국대 “학생회 주관 교외 OT·MT 금지”,ㆍ‘신입생 성추행’ 재발 방지책 ㆍ교수가 동행하는 MT는 허용 신입생 오리엔테이션에...,2016-03-02,2016
4,‘억대 뒷돈’ KT&G 간부 구속영장,KT&G 비리 의혹을 수사 중인 서울중앙지검 특수2부는 7일 KT&G와 거래해 온 ...,2016-03-07,2016


### 2. 컬렉션 만들고 데이터 적재

In [83]:
import chromadb
import os
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction  # chromaDB 자체에서 Openai 임베딩 모델 있음
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

embeddings = OpenAIEmbeddingFunction(
    model_name="text-embedding-3-small"
)

embed_client = chromadb.Client()

# collection은 
collection = embed_client.get_or_create_collection(
    name="criminal_collection",
    embedding_function=embeddings,
    configuration={'hnsw' : {"space" : "cosine"}}

)

In [88]:
ids = df_sampled.index.astype(str).tolist()
metadata = df_sampled[['제목', '일자', '연도']].to_dict('records')
# docs = df_sampled['본문'].tolist()
docs = df_sampled['본문'].astype(str).str[:6000].tolist()

In [85]:
len(docs[148])

1021

In [89]:
collection.upsert(ids=ids, documents=docs, metadatas=metadata)  # id 잘 안넣으면 랜덤으로 하기 때문에 잘 지정해줘야?
collection.count()

240

### 3. 검색 테스트

In [92]:
question = "2019년에 발생한 살인 사건에 대한 정보 알려줘"

# result = collection.query(query_texts=[question], n_results=5)
result = collection.query(query_texts=[question], n_results=5,
                          where={"연도" : {"$eq" : 2019}})  
result

{'ids': [['106', '107', '118', '97', '110']],
 'embeddings': None,
 'documents': [['전 남편을 살해한 혐의를 받는 고유정이 의붓아들이 숨진 날 인터넷 커뮤니티에 어린이를 위한 행사 개최를 제안한 것으로 드러났습니다. 청주 상당경찰서는 제주에서 넘겨받은 고 씨의 휴대전화 3대와 컴퓨터 하드디스크 2대를 정밀 분석해 이 같은 사실을 확인했다고 밝혔습니다. 경찰은 고 씨의 의붓아들 4살 A 군이 숨진 지난 3월 고 씨가 주변인과 나눈 대화와 인터넷 커뮤니티 작성 글, 검색 기록 등을 자세하게 들여다보고 있습니다. 조사 결과 고 씨는 A 군이 숨진 날인 지난 3월 2일 새벽 자신이 거주하는 아파트 인터넷 커뮤니티에 어린이들을 위한 행사를 열자고 제안한 것으로 드러났습니다. 경찰은 고 씨의 이런 행동이 A 군의 죽음과 연관성이 있는지 수사를 벌일 예정입니다. 앞서 고 씨의 재혼 남편인 37살 B 씨는 부인인 고 씨가 자신의 아들을 숨지게 한 정황이 있다며 검찰에 고소장을 제출했습니다. [jongkyu87',
   '별거 중인 아내를 잠복 후 살해한 ‘구월동 살인사건’의 범인 A씨에게 징역 25년의 형이 확정됐다. 대법원 1부는 살인 혐의로 기소된 A씨의 상고심에서 징역 25년을 선고한 원심판결을 확정했다고 24일 밝혔다. 대법원은 “죄질을 살펴본 결과 징역 25년이 마땅하다”고 판시했다. A씨는 지난해 7월13일 오후 8시15분쯤 인천시 남동구 구월동 한 주택가에서 미리 준비한 흉기로 아내 B씨의 복부 등을 수차례 찔러 살해했다. 당시 A씨와 B씨는 별거 후 이혼 소송을 진행 중이었다. A씨는 별거 후 B씨의 거주지를 몰랐다. A씨는 범행 당일 학교를 마치고 귀가하는 자녀들을 미행해 B씨의 집 앞에서 잠복했고 밖으로 나온 아내를 살해했다. 같은 해 11월23일 열린 결심공판에서 검찰은 A씨에 징역 26년을 구형했다. 당시 검찰은 “피고인은 한 달 전부터 피해자를 살해할 마음을 먹고 아내가 나오길 기다리며

In [97]:
context = "\n\n".join(result['documents'][0])   # 리스트 내용들을 \n\n과 함께 합쳐주는 기능! join 유용하게 잘 쓰자
print(context)

전 남편을 살해한 혐의를 받는 고유정이 의붓아들이 숨진 날 인터넷 커뮤니티에 어린이를 위한 행사 개최를 제안한 것으로 드러났습니다. 청주 상당경찰서는 제주에서 넘겨받은 고 씨의 휴대전화 3대와 컴퓨터 하드디스크 2대를 정밀 분석해 이 같은 사실을 확인했다고 밝혔습니다. 경찰은 고 씨의 의붓아들 4살 A 군이 숨진 지난 3월 고 씨가 주변인과 나눈 대화와 인터넷 커뮤니티 작성 글, 검색 기록 등을 자세하게 들여다보고 있습니다. 조사 결과 고 씨는 A 군이 숨진 날인 지난 3월 2일 새벽 자신이 거주하는 아파트 인터넷 커뮤니티에 어린이들을 위한 행사를 열자고 제안한 것으로 드러났습니다. 경찰은 고 씨의 이런 행동이 A 군의 죽음과 연관성이 있는지 수사를 벌일 예정입니다. 앞서 고 씨의 재혼 남편인 37살 B 씨는 부인인 고 씨가 자신의 아들을 숨지게 한 정황이 있다며 검찰에 고소장을 제출했습니다. [jongkyu87

별거 중인 아내를 잠복 후 살해한 ‘구월동 살인사건’의 범인 A씨에게 징역 25년의 형이 확정됐다. 대법원 1부는 살인 혐의로 기소된 A씨의 상고심에서 징역 25년을 선고한 원심판결을 확정했다고 24일 밝혔다. 대법원은 “죄질을 살펴본 결과 징역 25년이 마땅하다”고 판시했다. A씨는 지난해 7월13일 오후 8시15분쯤 인천시 남동구 구월동 한 주택가에서 미리 준비한 흉기로 아내 B씨의 복부 등을 수차례 찔러 살해했다. 당시 A씨와 B씨는 별거 후 이혼 소송을 진행 중이었다. A씨는 별거 후 B씨의 거주지를 몰랐다. A씨는 범행 당일 학교를 마치고 귀가하는 자녀들을 미행해 B씨의 집 앞에서 잠복했고 밖으로 나온 아내를 살해했다. 같은 해 11월23일 열린 결심공판에서 검찰은 A씨에 징역 26년을 구형했다. 당시 검찰은 “피고인은 한 달 전부터 피해자를 살해할 마음을 먹고 아내가 나오길 기다리며 잠복하는 등 매우 치밀하고 계획적으로 범행했다. 장기간 사회에서 격리돼 참회할 시간이 필요하다”고 구형 이유를 설명했다. A씨는 경찰조사 당시 “아내가 아픈 

In [119]:
question = "2019년에 발생한 살인 사건에 대한 정보 알려줘"

result_nowhere = collection.query(query_texts=[question], n_results=5)
# result = collection.query(query_texts=[question], n_results=5,
#                           where={"연도" : {"$eq" : 2019}})  
result_nowhere

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}} in query.

### 4. ChromaDB 저장하기

In [93]:
persistent_client = chromadb.PersistentClient(path="./criminal_db")

In [94]:
saved_collection = persistent_client.get_or_create_collection(
    name="criminal_collection",
    embedding_function=embeddings,
    configuration={"hnsw" : {"space" : "cosine"}}
)

In [95]:
saved_collection.upsert(ids=ids,
                        documents=docs,
                        metadatas=metadata)

### 5. 불러와서 테스트

In [101]:
load_client = chromadb.PersistentClient(path="criminal_db")
load_collection = load_client.get_collection(
    name="criminal_collection",
    embedding_function=embeddings   # configuration 어차피 안바뀌니 적을 필요 X, 임베딩도 또 하는건 아니지만 뭘 썼는지 알려주기 위해
)

In [106]:
question_load = "비리 사건은 언제 가장 많이 일어났어?"

result_filtered = load_collection.query(query_texts=[question_load], n_results=5,
                                        where_document={"$contains" : "비리"})
result_filtered

{'ids': [['139', '232', '4', '29', '52']],
 'embeddings': None,
 'documents': [["조국 전 법무부 장관이 14일 서울중앙지법에서 열린 ‘유재수 감찰무마 의혹’ 사건 5차 공판에 출석하며 취재진의 질문을 받고 있다. 청와대 민정수석으로 재직할 당시 유재수 전 부산시 경제부시장에 대한 ‘감찰 무마’ 의혹으로 재판을 받고 있는 조국 전 법무부 장관이 연일 ‘검찰 때리기’에 나섰다. 사회관계망서비스에서 검찰 내 성추행 사건과 접대 의혹 등 비리를 언급하며 고위공직자범죄수사처의 필요성을 주장하는가 하면, 앞서 재판에 출석하면서는 “검찰이 내부 비리에 솜방망이조차 들지 않았다”고 말하기도 했다. 조 전 장관은 15일 자신의 페이스북에 올린 글에서 “2015년 4월 서울남부지검 검사 재직시 후배 여검사 2명에게 성폭력을 가했으나 아무 징계나 처벌 없이 사건 발생 다음 날 사직 처리되고 같은 해 CJ 임원으로 취업한 사람이 ‘누구’인지, 이 ‘누구’가 누구 아들인지, 그리고 이 ‘누구’의 매형이 누구인지 다 아시죠?”라고 물었다. 그는 “2018년 1월 서지현 검사의 용기있는 문제 제기로 사회적 파문이 일어나고 ‘검찰 성추행 사건의 진상규명과 피해회복을 위한 진상조사단’이 만들어져 조사를 한 후 이 ‘누구’는 비로소 불구속 기소됐다”고 덧붙였다. 이어 조 전 장관은 “폭로 후 서 검사는 검찰 조직 내에서 “조직 부적응자’ 취급을 받으며 ‘왕따’가 됐다”며 “그리고 검찰 구성원들은 서 검사에 대한 부정적 인상을 주는 단편적 사실을 언론에 흘렸다”고 주장했다. 조 전 장관은 “공수처가 왜 필요한지 보여주는 단적인 사례”라며 “이 ‘누구' 외에도 유사한 사례가 많았다는 점, 첨언한다”고 밝혔다. 그러면서 조 전 장관은 “언론에서 보도를 하지 않아 다 묻혔을 뿐”이라고 강조했다. 조 전 장관이 언급한 ‘누구’는 진모 전 검사다. 진 전 검사는 2015년 회식 자리에서 술에 취한 후배 검사 2명을 성추행한 혐의로 기소돼 

In [107]:
context_load = "\n\n".join(result_filtered['documents'][0])   # 리스트 내용들을 \n\n과 함께 합쳐주는 기능! join 유용하게 잘 쓰자
print(context_load)

조국 전 법무부 장관이 14일 서울중앙지법에서 열린 ‘유재수 감찰무마 의혹’ 사건 5차 공판에 출석하며 취재진의 질문을 받고 있다. 청와대 민정수석으로 재직할 당시 유재수 전 부산시 경제부시장에 대한 ‘감찰 무마’ 의혹으로 재판을 받고 있는 조국 전 법무부 장관이 연일 ‘검찰 때리기’에 나섰다. 사회관계망서비스에서 검찰 내 성추행 사건과 접대 의혹 등 비리를 언급하며 고위공직자범죄수사처의 필요성을 주장하는가 하면, 앞서 재판에 출석하면서는 “검찰이 내부 비리에 솜방망이조차 들지 않았다”고 말하기도 했다. 조 전 장관은 15일 자신의 페이스북에 올린 글에서 “2015년 4월 서울남부지검 검사 재직시 후배 여검사 2명에게 성폭력을 가했으나 아무 징계나 처벌 없이 사건 발생 다음 날 사직 처리되고 같은 해 CJ 임원으로 취업한 사람이 ‘누구’인지, 이 ‘누구’가 누구 아들인지, 그리고 이 ‘누구’의 매형이 누구인지 다 아시죠?”라고 물었다. 그는 “2018년 1월 서지현 검사의 용기있는 문제 제기로 사회적 파문이 일어나고 ‘검찰 성추행 사건의 진상규명과 피해회복을 위한 진상조사단’이 만들어져 조사를 한 후 이 ‘누구’는 비로소 불구속 기소됐다”고 덧붙였다. 이어 조 전 장관은 “폭로 후 서 검사는 검찰 조직 내에서 “조직 부적응자’ 취급을 받으며 ‘왕따’가 됐다”며 “그리고 검찰 구성원들은 서 검사에 대한 부정적 인상을 주는 단편적 사실을 언론에 흘렸다”고 주장했다. 조 전 장관은 “공수처가 왜 필요한지 보여주는 단적인 사례”라며 “이 ‘누구' 외에도 유사한 사례가 많았다는 점, 첨언한다”고 밝혔다. 그러면서 조 전 장관은 “언론에서 보도를 하지 않아 다 묻혔을 뿐”이라고 강조했다. 조 전 장관이 언급한 ‘누구’는 진모 전 검사다. 진 전 검사는 2015년 회식 자리에서 술에 취한 후배 검사 2명을 성추행한 혐의로 기소돼 1심에서 징역 10개월을 선고받았다. 그러나 진 전 검사는 검찰에서 처벌이나 징계 절차가 이뤄지지 않은 채 사표가 수리됐고, 대기업 법무담당 임원

### 6. context에 포함하여 최종 요청

In [110]:
import numpy as np
def embed(texts):
    """텍스트 목록을 받아서 벡터 배열로 변환하는 함수. openai embedding을 사용해 문서 수 x 1536차원으로 변환"""

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return np.array([item.embedding for item in response.data])

In [ ]:
# 먼저 docs 임베딩
doc_vecs = embed(docs)
doc_vecs[0]

array([-0.00741959,  0.02894592, -0.04403687, ..., -0.02981567,
        0.009758  ,  0.00695419], shape=(1536,))

#### 같은 질문에 대해 관련문서 있고 없고 비교

In [112]:
question = "대장동 비리 일환에 대한 검찰 수사 내용"

# 테스트용으로 근거 없이 답변 받아보기

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role" : "user", "content" : question}]
)

print(response.choices[0].message.content)

대장동 개발 비리 사건은 성남시 대장동 민관합동 개발 과정에서 **민간업자들이 과도한 개발이익을 얻고, 성남도시개발공사와 성남시에 손해를 끼쳤는지**를 둘러싼 검찰 수사·재판 사건입니다.

### 1. 검찰이 문제 삼은 핵심 구조
- 성남도시개발공사가 사업에 참여하면서 민간사업자에게 유리한 수익 배분 구조를 설정했다는 의혹
- 화천대유 및 관계자들이 적은 자본으로 막대한 배당이익을 얻었다는 의혹
- 사업 공모·심사 과정에서 특정 민간업자에게 편의를 제공했다는 의혹
- 그 대가로 사업 관계자들에게 금품이나 정치자금이 제공됐다는 의혹

### 2. 주요 피고인·수사 대상
- **김만배**: 화천대유 대주주. 배임, 뇌물, 범죄수익 은닉 등 혐의
- **남욱**: 천화동인 4호 소유자. 배임 및 불법 정치자금 관련 혐의
- **정영학**: 천화동인 5호 소유자. 사업 구조와 자금 흐름을 보여주는 녹취·자료를 검찰에 제출
- **유동규**: 전 성남도시개발공사 본부장. 민간업자와의 유착, 뇌물·배임 혐의
- **정진상**: 이재명 당시 성남시장 측근. 뇌물 및 부정처사 관련 혐의
- **김용**: 이재명 측근. 대장동 민간업자들로부터 불법 정치자금을 받은 혐의
- **이재명**: 성남시장 재직 당시 사업 결정과 관련해 배임·이해충돌·뇌물 등 혐의로 기소됨. 이 부분은 본인이 혐의를 부인하고 있으며 재판에서 다투는 사안입니다.

### 3. 검찰의 주요 주장
검찰은 대체로 다음과 같이 주장해 왔습니다.

1. 성남시와 성남도시개발공사가 공공개발의 위험은 부담하면서도 초과이익 환수 장치를 충분히 두지 않았다.  
2. 그 결과 민간업자들에게 수천억 원대의 이익이 돌아갔다.  
3. 사업 관계자들 사이에 사전에 역할을 나눈 정황과 금품 거래가 있었다.  
4. 개발이익 일부가 정치자금이나 뇌물로 제공됐다는 것입니다.

### 4. 피고인 측 반론
피고인 측은 다음과 같이 반박해 왔습니다.

- 사업 당시 민간사업자 선정과 수익 배분은 적법한 절차에 따라 이뤄졌다.
- 당시

In [113]:
query_vec = embed([question])[0]
similarity = doc_vecs @ query_vec

top = pd.Series(similarity).sort_values(ascending=False).head(5)

for i, score in top.items():
    print (i, score, docs[i][:50])

232 0.40677357175047746 '국민 특검'으로 불렸던 박영수 전 특검이 구속되며 대장동 비리 일환인 '50억 클럽'에 
142 0.4063321919846601 이동재 전 채널 A기자의 강요미수 사건을 수사했던 정진웅 광주지검 차장 검사가 16일 서울
147 0.4061278940485522 한동훈 검사장과 정진웅 차장검사. 서울고등검찰청이 ‘채널A 기자 강요미수 의혹’ 관련 한동
66 0.40226861567063565 ■ 검찰이 100억 원대 뇌물수수 의혹 등을 받는 이명박 전 대통령에게 오는 14일 소환을
208 0.3996004093812928 이태원 참사를 수사하고 있는 특별수사본부가 이상민 행정안전부 장관을 본격적으로 수사할지 검


In [114]:
context = ""

for i in top.index:
    context += docs[i] + "\n\n"

print(context) 

'국민 특검'으로 불렸던 박영수 전 특검이 구속되며 대장동 비리 일환인 '50억 클럽'에 대한 검찰 수사도 한층 탄력을 받게 됐습니다. 곽상도 전 국민의 힘 의원 등 여러 인물이 관련 의혹에 연루돼 있는데요. 누가 본격적으로 수사 선상에 오를지 기자가 짚어봤습니다. 구속된 박영수 전 특검 다음으로 '50억 클럽'에서 수사가 진척된 건 곽상도 전 국민의힘 의원입니다. 검찰은 곽 전 의원이 대장동 일당의 청탁을 받고 하나금융지주에 영향력을 행사해주는 대가로, 아들을 통해 화천대유로부터 뇌물 50억 원을 받았다고 보고 있습니다. 그러나 지난 2월 1심 재판부는 곽 전 의원이 실제로 영향력을 행사했는지, 아들이 받은 50억 원이 곽 전 의원이 받은 뇌물로 볼 수 있는지 의문이라며 무죄를 선고했습니다. 이후 재수사에 나선 검찰은 최근, 공범인 아들 병채 씨를 두 차례 불러 조사했습니다. 항소심 재판으로 피고인 신분인 곽 전 의원을 다시 불러 조사하기가 어려운 만큼, 핵심 피의자인 병채 씨를 강도 높게 조사해 혐의 다지기에 주력하는 모습입니다. 곽 전 의원 다음으로는 권순일 전 대법관이 수사 대상으로 꼽힙니다. 권 전 대법관은 대법관 재임 중이던 2020년 7월 이재명 민주당 대표가 공직선거법 위반 재판에서 무죄를 받는 데 힘을 썼다는 의혹에 휩싸여 있습니다. 퇴임 후엔 화천대유 고문으로 취업해 1억5천만 원을 고문료로 받은 사실이 드러나 논란이 됐습니다. 검찰은 권 전 대법관을 대장동 수사 초기인 재작년 11월과 12월 두 차례 불러 조사하고는 추가 조치를 하지 않았습니다. 다만 곽 전 의원 재수사·박 전 특검 구속을 계기로 권 전 대법관에 대한 수사도 본격화할 거라는 게 검찰 안팎의 전망입니다. 대장동 의혹이 불거진 이후 김만배 씨와 만나 대책을 논의하고 검사 출신 변호사를 소개한 것으로 조사된 김수남 전 검찰총장 역시 수사 선상에 있습니다. 검찰은 50억 클럽 의혹 전반을 수사한다는 게 기본 입장이라며, 관련된 인물들을 여러 방식으로, 차례대로 수사할 거라고 강조했

In [115]:
prompt = f"""
아래 [근거 자료]만 참고해서 질문에 답하세요.
자료에 없으면 '자료에 없음' 이라고 답하세요

[근거 자료]
{context}

[질문]
{question}
"""

response_rag = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role" : "user", "content" : prompt}]    # 질문 prompt 전부 넣음!
)

print(response_rag.choices[0].message.content)

대장동 비리의 일환인 **‘50억 클럽’ 의혹**에 대해 검찰은 다음과 같이 수사하고 있습니다.

- **박영수 전 특별검사**: 구속되면서 50억 클럽 수사가 탄력을 받고 있습니다.
- **곽상도 전 국민의힘 의원**: 대장동 일당의 청탁을 받고 하나금융지주에 영향력을 행사한 대가로 아들을 통해 화천대유에서 50억 원을 받았다는 의혹을 받고 있습니다. 1심에서는 무죄가 선고됐지만, 검찰은 재수사에 착수해 아들 병채 씨를 두 차례 불러 조사하며 혐의 입증에 주력하고 있습니다.
- **권순일 전 대법관**: 이재명 대표의 공직선거법 위반 사건 무죄 판결에 힘을 썼다는 의혹과, 퇴임 후 화천대유 고문으로 일하며 1억 5천만 원을 받은 사실과 관련해 수사 대상에 거론되고 있습니다. 과거 두 차례 조사를 받았으며, 박영수 전 특검 구속 등을 계기로 수사가 본격화될 가능성이 제기됐습니다.
- **김수남 전 검찰총장**: 대장동 의혹이 불거진 뒤 김만배 씨와 만나 대책을 논의하고 검사 출신 변호사를 소개한 것으로 조사돼 수사 선상에 있습니다.

검찰은 ‘50억 클럽’ 의혹 전반을 수사한다는 입장으로, 관련 인물들을 여러 방식으로 차례대로 조사할 계획이라고 밝혔습니다.


#### 관련문서 없을 경우 대처 테스트

In [117]:
question3 = "2010년 서울에서 일어난 살인사건"

prompt = f"""
아래 [근거 자료]만 참고해서 질문에 답하세요.
자료에 없으면 '자료에 없음' 이라고 답하세요

[근거 자료]
{context}

[질문]
{question3}
"""

response_rag = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role" : "user", "content" : prompt}]    # 질문 prompt 전부 넣음!
)

print(response_rag.choices[0].message.content)

자료에 없음


### Troubleshooting 정리

1. 본문에 200개 정도 NaN이 있어서 오류 
    - dropna로 수정
2. 연도별 100개 했더니 전체 용량 제한 -
    - 결국 연도별 30개, 총 문서 240개만
3. docs의 149번째 기사(input[148]) 한 개가 8,192토큰 제한을 넘음 
    - 임베딩용 docs를 만들 때 모든 본문을 예를 들어 앞 6,000자까지만 사용하기로.!
4. query에 where나 where_document 필터링 없이 그냥 질문만으로 '2019년 강도 사건' 검색했더니, 2019년도 강도도 아닌 다른 기사들 나옴
    - 필터링 사용하기로

- 추가 사항
    - 초반 6000자만 사용하지 말고, 길면 두 문서로 나누는 방법도 다음에 해 보기!
    - 같은 질문에 대해 where 필터링 있고 없고 결과 비교해서 질문만으로 어느 정도 필터링되는지 분석하려 했는데, credit 남지 않았다고 해서 더 이상의 분석은 할 수 없었다....